# Galaxy Merger Tracking — SIMBA cis100

Runs the full merger-detection pipeline on real SIMBA-100 data.

**Prerequisites**
- `simbanator` registered for `cis100` in `~/.simbanator/config.json`
- Progenitor track file at `output/cis100/progenitors/progenitors_most_mass.fits`  
  (produced by `caesar_read_progen` — see `test_sfh_caesar.ipynb`)

**Pipeline**
1. Load simulation handle and progenitor track file  
2. Run `process_galaxies_with_tracks` — find companions in each snapshot  
3. Run `analyze_mergers` — classify major / minor events  
4. Visualise results

## 1 · Setup

In [ ]:
import sys, os
_pkg_root = '/home/lorenzong/analize_simba_cgm'
if _pkg_root not in sys.path:
    sys.path.insert(0, _pkg_root)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

from simbanator.io.simba import Simulation
from simbanator.io.paths import OutputPaths
from simbanator.analysis.mergers import (
    process_galaxies_with_tracks,
    analyze_mergers,
)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

In [ ]:
# ── merger-detection parameters ──────────────────────────────────────────
BOX_SIZE             = 100.0   # Mpc/h — SIMBA 100 box
SEARCH_RADIUS_FACTOR = 5.0     # search sphere = factor × r_half
MASS_THRESHOLD       = 1e9     # min companion stellar mass (M_sun)
RHALF_UNIT_FACTOR    = 1e-3    # kpc/h → Mpc/h (Caesar default units)

MASS_RATIO_MAJOR     = 0.25    # >= major merger
MASS_RATIO_MINOR     = 0.10    # >= minor merger (< major)

## 2 · Simulation handle and track file

In [ ]:
sim = Simulation('cis100')
out = OutputPaths(sim.name)

track_path = os.path.join(out.progenitors, 'progenitors_most_mass.fits')
print('Track file :', track_path)
print('Exists     :', os.path.exists(track_path))

In [ ]:
# Inspect the track file structure
with fits.open(track_path) as hdul:
    hdul.info()
    colnames = list(hdul[1].columns.names)
    n_rows   = len(hdul[1].data)

# Column names are snapshot numbers (plus 'GroupID' written by caesar_read_progen)
snap_cols = [c for c in colnames if c.upper() != 'GROUPID']
snaplist  = sorted([int(c) for c in snap_cols])   # ascending: 44, 45, … 150

print(f'\nTracked galaxies : {n_rows}')
print(f'Snapshot columns : {len(snap_cols)}')
print(f'Snapshot range   : {snaplist[0]} → {snaplist[-1]}')
print(f'All columns      : {colnames[:4]} … {colnames[-3:]}')

### Snapshot → redshift mapping
Loaded from the bundled scale-factor table (no live catalog access needed).

In [ ]:
# Load the bundled snap→redshift table for SIMBA-100
_data_dir = os.path.join(_pkg_root, 'simbanator', 'data', 'snap_z_maps')
scale_factors = np.loadtxt(os.path.join(_data_dir, 'zsnap_map_caesar_box100.txt'))

def snap_to_z(s):
    return 1.0 / scale_factors[int(s)] - 1.0

redshifts = np.array([snap_to_z(s) for s in snaplist])
print(f'z range: {redshifts.min():.3f} (snap {snaplist[0]}) → '
      f'{redshifts.max():.3f} (snap {snaplist[-1]})')

## 3 · Find merger companions

`process_galaxies_with_tracks` reads the Caesar HDF5 catalogs directly and records all
neighbours within `SEARCH_RADIUS_FACTOR × r_half` as `Progenitor` entries.

> **Note on track file format**  
> `caesar_read_progen` writes a `GroupID` column plus one column per snapshot  
> (named by snapshot number, e.g. `'44'`, `'45'`, …).  
> The updated `process_galaxies_with_tracks` now handles this automatically —  
> it drops `GroupID` and sorts columns by snapshot number.

In [ ]:
# Build catalog paths in the same ascending order as the track file columns
caesar_paths = [sim.get_caesar_file(s) for s in snaplist]

galaxies = process_galaxies_with_tracks(
    track_fits_path      = track_path,
    box_size             = BOX_SIZE,
    caesar_paths         = caesar_paths,
    search_radius_factor = SEARCH_RADIUS_FACTOR,
    mass_threshold       = MASS_THRESHOLD,
    rhalf_unit_factor    = RHALF_UNIT_FACTOR,
)

In [ ]:
total_progs = sum(len(g.progenitors) for g in galaxies.values())
gals_with_progs = sum(1 for g in galaxies.values() if g.progenitors)

print(f'Tracked galaxies          : {len(galaxies)}')
print(f'Galaxies with companions  : {gals_with_progs}')
print(f'Total companion entries   : {total_progs}')

### Diagnostic — search radius sanity check
Verify that the search radii are physically reasonable before trusting the companion counts.

In [ ]:
# Sample the search radius from the Galaxy objects (set at snap_idx=0, i.e. snap 44)
r_halfs = np.array([g.r for g in galaxies.values() if g.r > 0])
if len(r_halfs):
    r_search_kpc = r_halfs * SEARCH_RADIUS_FACTOR        # kpc/h (before unit conversion)
    r_search_mpc = r_search_kpc * RHALF_UNIT_FACTOR      # Mpc/h
    print(f'r_half range (kpc/h)        : {r_halfs.min():.1f} – {r_halfs.max():.1f}')
    print(f'Search radius range (kpc/h) : {r_search_kpc.min():.1f} – {r_search_kpc.max():.1f}')
    print(f'Search radius range (Mpc/h) : {r_search_mpc.min():.4f} – {r_search_mpc.max():.4f}')
else:
    print('No r_half values stored — galaxies were absent at snap 44 (first column).')

## 4 · Classify mergers

In [ ]:
n_snaps = len(snaplist)
n_gals  = len(galaxies)

major_mergers, minor_mergers = analyze_mergers(
    galaxies,
    array_size         = (n_snaps, n_gals),
    mass_threshold_maj = MASS_RATIO_MAJOR,
    mass_threshold_min = MASS_RATIO_MINOR,
)

print('Array shape (n_snaps, n_gals):', major_mergers.shape)
print(f'Total major merger events : {major_mergers.sum()}')
print(f'Total minor merger events : {minor_mergers.sum()}')

In [ ]:
# Per-galaxy merger counts (top 20 by total events)
total_per_gal = major_mergers.sum(axis=0) + minor_mergers.sum(axis=0)
top_idx = np.argsort(total_per_gal)[::-1][:20]

print(f'{"Galaxy idx":>12}  {"Major":>8}  {"Minor":>8}  {"Total":>8}')
print('-' * 44)
for idx in top_idx:
    maj = int(major_mergers[:, idx].sum())
    mni = int(minor_mergers[:, idx].sum())
    if maj + mni == 0:
        break
    print(f'{idx:>12}  {maj:>8}  {mni:>8}  {maj+mni:>8}')

## 5 · Visualisation

In [ ]:
# Merger rate vs redshift (sample-total per snapshot)
fig, ax = plt.subplots(figsize=(9, 4))

ax.step(redshifts, major_mergers.sum(axis=1), where='mid',
        color='firebrick', lw=2, label='Major (ratio ≥ 0.25)')
ax.step(redshifts, minor_mergers.sum(axis=1), where='mid',
        color='steelblue', lw=2, label='Minor (0.10 ≤ ratio < 0.25)')

ax.set_xlabel('Redshift z')
ax.set_ylabel('Number of merger events')
ax.set_title('Merger count per snapshot (all tracked galaxies)')
ax.invert_xaxis()
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Per-galaxy merger history heatmap (only galaxies that have at least one event)
active = np.where((major_mergers.sum(axis=0) + minor_mergers.sum(axis=0)) > 0)[0]

if len(active) == 0:
    print('No merger events found — nothing to plot.')
else:
    combined = np.zeros((n_snaps, len(active)), dtype=int)
    combined[minor_mergers[:, active] > 0] = 1
    combined[major_mergers[:, active] > 0] = 2

    fig, ax = plt.subplots(figsize=(11, max(3, len(active) * 0.25 + 1)))
    cmap = plt.cm.get_cmap('RdYlGn_r', 3)
    im = ax.imshow(combined.T, aspect='auto', origin='lower',
                   cmap=cmap, vmin=-0.5, vmax=2.5,
                   extent=[redshifts[0], redshifts[-1], -0.5, len(active) - 0.5])

    cbar = fig.colorbar(im, ax=ax, ticks=[0, 1, 2], pad=0.02)
    cbar.ax.set_yticklabels(['None', 'Minor', 'Major'])

    ax.set_xlabel('Redshift z')
    ax.set_ylabel('Galaxy index (active only)')
    ax.set_title(f'Merger history — {len(active)} galaxies with at least one event')
    ax.set_yticks(range(len(active)))
    ax.set_yticklabels(active)
    plt.tight_layout()
    plt.show()

In [ ]:
# Mass-ratio distribution of all recorded companions
all_mass_ratios = np.concatenate([
    gal.get_attribute_values('mass')
    for gal in galaxies.values()
    if gal.progenitors
])

if len(all_mass_ratios) == 0:
    print('No companions recorded — skipping mass-ratio plot.')
else:
    fig, ax = plt.subplots(figsize=(7, 4))
    bins = np.linspace(0, 1, 30)
    ax.hist(all_mass_ratios, bins=bins, color='slategray', edgecolor='white', lw=0.4)
    ax.axvline(MASS_RATIO_MAJOR, color='firebrick', ls='--', lw=1.5,
               label=f'Major threshold ({MASS_RATIO_MAJOR})')
    ax.axvline(MASS_RATIO_MINOR, color='steelblue', ls='--', lw=1.5,
               label=f'Minor threshold ({MASS_RATIO_MINOR})')
    ax.set_xlabel(r'Stellar mass ratio  $m_{\rm sec}\,/\,m_{\rm main}$')
    ax.set_ylabel('Count')
    ax.set_title('Mass-ratio distribution of merger companions')
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Companion separation distribution
all_distances = np.concatenate([
    gal.get_attribute_values('distance')
    for gal in galaxies.values()
    if gal.progenitors
])

if len(all_distances) == 0:
    print('No companions recorded — skipping distance plot.')
else:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(all_distances, bins=40, color='mediumpurple', edgecolor='white', lw=0.4)
    ax.set_xlabel('Separation [Mpc/h]')
    ax.set_ylabel('Count')
    ax.set_title('Distribution of companion separations')
    plt.tight_layout()
    plt.show()

## 6 · Save results

In [ ]:
import h5py

save_dir  = out.subdir('mergers')
save_path = os.path.join(save_dir, 'merger_arrays.hdf5')

with h5py.File(save_path, 'w') as f:
    f.create_dataset('major_mergers', data=major_mergers)
    f.create_dataset('minor_mergers', data=minor_mergers)
    f.create_dataset('snaplist',      data=np.array(snaplist))
    f.create_dataset('redshifts',     data=redshifts)
    f.attrs['n_galaxies']            = n_gals
    f.attrs['search_radius_factor']  = SEARCH_RADIUS_FACTOR
    f.attrs['mass_threshold_msun']   = MASS_THRESHOLD
    f.attrs['mass_ratio_major']      = MASS_RATIO_MAJOR
    f.attrs['mass_ratio_minor']      = MASS_RATIO_MINOR

print('Saved to:', save_path)

## 7 · Trajectory Diagnostic — do detected companions actually merge?

For one selected galaxy (the one with the most recorded companions), we:

1. **Reconstruct the main galaxy trajectory** by re-reading each Caesar HDF5 file directly.
2. **Plot companion separations vs. redshift** against the dynamic search radius — companions that truly merge should approach zero; flybys will not.
3. **Plot companion positions in the main-galaxy frame** — a merging companion should spiral inward.
4. **Multi-snapshot neighborhood panels** — show *all* catalog galaxies near the main galaxy at each detection snapshot; red = flagged by the merger tracker, blue = in context but not flagged.

A correct merger tracker should produce: (a) separations decreasing toward zero before the companion disappears, (b) relative positions converging on the origin, and (c) no obvious bright neighbors systematically missed or spuriously included.

In [ ]:
import h5py

# Caesar HDF5 field paths (same constants as in mergers.py)
_H5_POS   = 'galaxy_data/pos'
_H5_SMASS = 'galaxy_data/dicts/masses.stellar'
_H5_RHALF = 'galaxy_data/dicts/radii.stellar_half_mass'

# ── select the galaxy with the most recorded companions ──────────────────────
# Change gal_idx manually to inspect a different galaxy.
gal_idx = max(galaxies, key=lambda k: len(galaxies[k].progenitors))
galaxy  = galaxies[gal_idx]
progs   = list(galaxy.progenitors.values())

print(f"Inspecting galaxy index  : {gal_idx}")
print(f"Companion entries        : {len(progs)}")
print(f"Detection snapshot idxs  : {sorted(set(p.snapshot for p in progs))}")

# ── reconstruct main galaxy position at every snapshot ──────────────────────
track = galaxy.track.astype(int)

main_pos   = np.full((len(snaplist), 3), np.nan)
main_mass  = np.full(len(snaplist), np.nan)
main_rhalf = np.full(len(snaplist), np.nan)

for si, cpath in enumerate(caesar_paths):
    idx = track[si]
    if idx < 0:
        continue
    with h5py.File(cpath, 'r') as f:
        n_cat = f[_H5_POS].shape[0]
        if idx >= n_cat:
            continue
        main_pos[si]   = f[_H5_POS][idx]
        main_mass[si]  = f[_H5_SMASS][idx]
        main_rhalf[si] = f[_H5_RHALF][idx]

valid = ~np.isnan(main_pos[:, 0])
search_radii_mpc = main_rhalf * SEARCH_RADIUS_FACTOR * RHALF_UNIT_FACTOR  # Mpc/h

print(f"\nTracked at {valid.sum()}/{len(snaplist)} snapshots")
print(f"z range covered: {redshifts[valid].min():.3f} – {redshifts[valid].max():.3f}")
print(f"Search radius range: {np.nanmin(search_radii_mpc)*1e3:.1f} – "
      f"{np.nanmax(search_radii_mpc)*1e3:.1f} kpc/h")

In [ ]:
# ── Plot A: companion separation vs. redshift ─────────────────────────────────
# Each dot = one progenitor entry.  Colour = mass ratio (bright = major).
# Red tick marks = search radius at that snapshot.
# A real merger should show separation monotonically approaching the tick.

snap_idx_p = np.array([p.snapshot for p in progs], dtype=int)
dist_kpc   = np.array([p.distance for p in progs]) * 1e3   # Mpc/h → kpc/h
mratio_p   = np.array([p.mass     for p in progs])
zed_p      = redshifts[snap_idx_p]
sr_kpc_p   = search_radii_mpc[snap_idx_p] * 1e3            # search radius at each snap

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, yscale in zip(axes, ('linear', 'log')):
    sc = ax.scatter(zed_p, dist_kpc, c=mratio_p, cmap='plasma',
                    vmin=0.1, vmax=1.0, s=55, zorder=4,
                    edgecolors='k', lw=0.3, label='detected companion')
    ax.scatter(zed_p, sr_kpc_p, marker='_', s=300, color='firebrick',
               zorder=5, label='search radius at snapshot')
    for z, d, sr in zip(zed_p, dist_kpc, sr_kpc_p):
        ax.plot([z, z], [d, sr], color='gray', lw=0.5, alpha=0.4, zorder=2)

    ax.set_xlabel('Redshift z', fontsize=12)
    ax.set_ylabel('Separation [kpc/h]', fontsize=12)
    ax.set_yscale(yscale)
    ax.invert_xaxis()
    ax.legend(fontsize=10)
    ax.set_title(f'Galaxy {gal_idx}  —  companion separations ({yscale} y)', fontsize=12)

cb = fig.colorbar(sc, ax=axes[1])
cb.set_label(r'$m_{\rm sec}\,/\,m_{\rm main}$', fontsize=11)

plt.suptitle('Plot A: Companion separation vs. redshift\n'
             'Separations should decrease toward zero before the companion disappears',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot B: companion positions in the main-galaxy rest frame ────────────────
# Each dot = one progenitor entry, colour = redshift (blue=high-z, red=low-z).
# Main galaxy sits at the origin.  Dashed circle = mean search radius.
# A merging companion should show dots spiralling toward (0, 0).
# A flyby crosses without converging.  A satellite orbits at fixed radius.

rel_x, rel_y, rel_z = [], [], []
mratio_b, zed_b = [], []

for p in progs:
    si = p.snapshot
    if np.isnan(main_pos[si, 0]):
        continue
    dx = p.x - main_pos[si, 0]
    dy = p.y - main_pos[si, 1]
    dz = p.z - main_pos[si, 2]
    # Minimum-image correction
    dx -= BOX_SIZE * np.round(dx / BOX_SIZE)
    dy -= BOX_SIZE * np.round(dy / BOX_SIZE)
    dz -= BOX_SIZE * np.round(dz / BOX_SIZE)
    rel_x.append(dx * 1e3)   # kpc/h
    rel_y.append(dy * 1e3)
    rel_z.append(dz * 1e3)
    mratio_b.append(p.mass)
    zed_b.append(redshifts[si])

rel_x, rel_y, rel_z = np.array(rel_x), np.array(rel_y), np.array(rel_z)
mratio_b, zed_b = np.array(mratio_b), np.array(zed_b)
mean_sr_kpc = np.nanmean(search_radii_mpc) * 1e3

fig, axes = plt.subplots(1, 3, figsize=(17, 6))
projections = [('x–y', rel_x, rel_y), ('x–z', rel_x, rel_z), ('y–z', rel_y, rel_z)]

for ax, (label, xa, ya) in zip(axes, projections):
    sc = ax.scatter(xa, ya, c=zed_b, cmap='coolwarm_r', s=55, zorder=4,
                    edgecolors='k', lw=0.3)
    ax.plot(0, 0, 'k*', ms=14, zorder=5, label='Main galaxy')
    circle = plt.Circle((0, 0), mean_sr_kpc, fill=False, color='firebrick',
                         ls='--', lw=1.5,
                         label=f'Mean search radius ({mean_sr_kpc:.0f} kpc/h)')
    ax.add_patch(circle)
    lim = mean_sr_kpc * 1.3
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect('equal')
    ax.set_xlabel(f'$\\Delta${label[0].lower()} [kpc/h]', fontsize=12)
    ax.set_ylabel(f'$\\Delta${label[-1].lower()} [kpc/h]', fontsize=12)
    ax.set_title(f'Projection {label}', fontsize=12)
    ax.legend(fontsize=9)

cb = fig.colorbar(sc, ax=axes[-1], fraction=0.046, pad=0.04)
cb.set_label('Redshift z', fontsize=11)

plt.suptitle(f'Plot B: Companion positions in main-galaxy frame — Galaxy {gal_idx}\n'
             'Colour = redshift (blue=high-z, red=low-z)',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot C: multi-snapshot neighborhood panels ────────────────────────────────
# For each snapshot where a companion was detected, load ALL catalog galaxies
# within CONTEXT_FACTOR × search_radius and plot them.
# Red  = flagged by merger tracker  |  Blue = present but not flagged
# Star = main galaxy  |  Dot size ∝ log stellar mass
# This is an independent check: do the red dots look like genuine companions?

from matplotlib.lines import Line2D

CONTEXT_FACTOR = 3.0   # show galaxies out to this × search radius

snaps_with_progs = sorted(set(p.snapshot for p in progs))
panel_snaps = snaps_with_progs[:min(9, len(snaps_with_progs))]

ncols = 3
nrows = int(np.ceil(len(panel_snaps) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 6 * nrows), squeeze=False)
axes = axes.ravel()

for ax_i, si in enumerate(panel_snaps):
    ax     = axes[ax_i]
    cpath  = caesar_paths[si]
    z      = redshifts[si]
    sr     = search_radii_mpc[si]   # Mpc/h
    mpos   = main_pos[si]           # (3,) Mpc/h

    if np.isnan(mpos[0]) or np.isnan(sr):
        ax.set_visible(False)
        continue

    context_r = CONTEXT_FACTOR * sr

    with h5py.File(cpath, 'r') as f:
        pos_all   = f[_H5_POS][:]
        smass_all = f[_H5_SMASS][:]

    # Minimum-image separations in x–y
    dx_all = pos_all[:, 0] - mpos[0]
    dy_all = pos_all[:, 1] - mpos[1]
    dz_all = pos_all[:, 2] - mpos[2]
    dx_all -= BOX_SIZE * np.round(dx_all / BOX_SIZE)
    dy_all -= BOX_SIZE * np.round(dy_all / BOX_SIZE)
    dz_all -= BOX_SIZE * np.round(dz_all / BOX_SIZE)
    dist3d = np.sqrt(dx_all**2 + dy_all**2 + dz_all**2)

    # Catalog indices within the context sphere above mass threshold
    ctx_mask = (dist3d < context_r) & (smass_all > MASS_THRESHOLD)
    ctx_idx  = np.where(ctx_mask)[0]

    # Build set of (x,y,z) positions flagged by tracker at this snapshot
    detected_xyz = set()
    for p in progs:
        if p.snapshot == si:
            detected_xyz.add((round(p.x, 4), round(p.y, 4), round(p.z, 4)))

    # Plot each context galaxy
    for ii in ctx_idx:
        key = (round(pos_all[ii, 0], 4), round(pos_all[ii, 1], 4), round(pos_all[ii, 2], 4))
        color  = 'firebrick' if key in detected_xyz else 'steelblue'
        log_m  = max(np.log10(smass_all[ii] + 1), 7.0)
        size   = (log_m - 7.0) * 25 + 15
        ax.scatter(dx_all[ii] * 1e3, dy_all[ii] * 1e3,
                   s=size, c=color, alpha=0.85, edgecolors='k', lw=0.3, zorder=4)

    # Main galaxy (idx removed from scatter already — dist3d[main_idx]=0 is within ctx but excluded)
    ax.plot(0, 0, 'k*', ms=14, zorder=6)

    # Search radius circle
    circle = plt.Circle((0, 0), sr * 1e3, fill=False,
                         color='firebrick', ls='--', lw=1.5, zorder=5)
    ax.add_patch(circle)

    n_detected = sum(1 for p in progs if p.snapshot == si)
    ax.set_aspect('equal')
    lim = context_r * 1e3 * 1.05
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_xlabel(r'$\Delta x$ [kpc/h]', fontsize=10)
    ax.set_ylabel(r'$\Delta y$ [kpc/h]', fontsize=10)
    ax.set_title(f'snap {snaplist[si]}  (z = {z:.3f})\n'
                 f'{n_detected} detected  |  search r = {sr*1e3:.0f} kpc/h', fontsize=10)

for ax in axes[len(panel_snaps):]:
    ax.set_visible(False)

legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='firebrick',
           markersize=10, markeredgecolor='k', label='detected by merger tracker'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='steelblue',
           markersize=10, markeredgecolor='k', label='in context, not detected'),
    Line2D([0], [0], marker='*', color='k', markersize=12, label='main galaxy'),
    Line2D([0], [0], color='firebrick', ls='--', lw=1.5, label='search radius'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=4,
           fontsize=11, bbox_to_anchor=(0.5, -0.03))

plt.suptitle(f'Plot C: Galaxy neighborhood at companion-detection snapshots — Galaxy {gal_idx}\n'
             f'(dot size ∝ log stellar mass)',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 8 · Single-galaxy trajectory + search-sphere companions (all snapshots in one plot)

The main galaxy's x–y path across every tracked snapshot is drawn as a connected line (★ markers).
At each snapshot all galaxies inside the search sphere are independently re-read from the Caesar HDF5
and plotted at their absolute box positions (● markers).
Both are coloured by redshift so you can read the time sequence at a glance.
Dashed circles show the dynamic search radius at each snapshot position.

In [ ]:
import h5py

_H5_POS   = 'galaxy_data/pos'
_H5_SMASS = 'galaxy_data/dicts/masses.stellar'
_H5_RHALF = 'galaxy_data/dicts/radii.stellar_half_mass'

# ── choose galaxy ────────────────────────────────────────────────────────────
# Default: pick galaxy with the most progenitor entries; change manually to test others.
PLOT_GAL_IDX = max(galaxies, key=lambda k: len(galaxies[k].progenitors))
pg  = galaxies[PLOT_GAL_IDX]
trk = pg.track.astype(int)

# ── collect main-galaxy positions and search radii from HDF5 ─────────────────
pos_main = np.full((len(snaplist), 3), np.nan)
sr_mpc   = np.full(len(snaplist), np.nan)

for si, cpath in enumerate(caesar_paths):
    idx = trk[si]
    if idx < 0:
        continue
    with h5py.File(cpath, 'r') as f:
        nc = f[_H5_POS].shape[0]
        if idx >= nc:
            continue
        pos_main[si] = f[_H5_POS][idx]
        sr_mpc[si]   = f[_H5_RHALF][idx] * SEARCH_RADIUS_FACTOR * RHALF_UNIT_FACTOR

# ── for each snapshot, independently find all companions in the search sphere ─
comp_x, comp_y, comp_si = [], [], []

for si, cpath in enumerate(caesar_paths):
    if np.isnan(pos_main[si, 0]) or np.isnan(sr_mpc[si]):
        continue
    mpos = pos_main[si]
    sr   = sr_mpc[si]

    with h5py.File(cpath, 'r') as f:
        pos_all   = f[_H5_POS][:]
        smass_all = f[_H5_SMASS][:]

    dx = pos_all[:, 0] - mpos[0]
    dy = pos_all[:, 1] - mpos[1]
    dz = pos_all[:, 2] - mpos[2]
    dx -= BOX_SIZE * np.round(dx / BOX_SIZE)
    dy -= BOX_SIZE * np.round(dy / BOX_SIZE)
    dz -= BOX_SIZE * np.round(dz / BOX_SIZE)
    dist = np.sqrt(dx**2 + dy**2 + dz**2)

    mask = (np.arange(len(smass_all)) != trk[si]) & (dist < sr) & (smass_all > MASS_THRESHOLD)
    for ii in np.where(mask)[0]:
        # Reconstruct absolute position with minimum-image shift applied
        comp_x.append(mpos[0] + dx[ii])
        comp_y.append(mpos[1] + dy[ii])
        comp_si.append(si)

comp_x  = np.array(comp_x)
comp_y  = np.array(comp_y)
comp_si = np.array(comp_si, dtype=int)

# ── plot ─────────────────────────────────────────────────────────────────────
valid_si = np.where(~np.isnan(pos_main[:, 0]))[0]
z_valid  = redshifts[valid_si]
z_min, z_max = z_valid.min(), z_valid.max()
norm = plt.Normalize(vmin=z_min, vmax=z_max)
cmap = plt.cm.plasma

fig, ax = plt.subplots(figsize=(9, 8))

# Search radius circles (low alpha to avoid clutter)
for si in valid_si:
    if np.isnan(sr_mpc[si]):
        continue
    circle = plt.Circle(
        (pos_main[si, 0], pos_main[si, 1]), sr_mpc[si],
        color=cmap(norm(redshifts[si])), fill=False, lw=0.8, alpha=0.25, ls='--', zorder=2,
    )
    ax.add_patch(circle)

# Main galaxy trajectory — break the line at periodic-boundary jumps
mx = pos_main[valid_si, 0]
my = pos_main[valid_si, 1]
jump = np.sqrt(np.diff(mx)**2 + np.diff(my)**2) > BOX_SIZE / 2
seg_start = 0
for k, is_jump in enumerate(jump):
    if is_jump:
        ax.plot(mx[seg_start:k+1], my[seg_start:k+1], 'k-', lw=0.7, alpha=0.35, zorder=3)
        seg_start = k + 1
ax.plot(mx[seg_start:], my[seg_start:], 'k-', lw=0.7, alpha=0.35, zorder=3)

sc_main = ax.scatter(mx, my, c=z_valid, cmap=cmap, norm=norm,
                     s=60, zorder=5, marker='*', edgecolors='k', lw=0.5,
                     label='Main galaxy')

# Companion positions
if len(comp_x):
    sc_comp = ax.scatter(comp_x, comp_y, c=redshifts[comp_si], cmap=cmap, norm=norm,
                         s=22, zorder=4, marker='o', alpha=0.75, edgecolors='k', lw=0.15,
                         label=f'Companions in search sphere ({len(comp_x)} total)')

cb = fig.colorbar(sc_main, ax=ax, pad=0.02)
cb.set_label('Redshift z', fontsize=11)

ax.set_xlabel('x [Mpc/h]', fontsize=12)
ax.set_ylabel('y [Mpc/h]', fontsize=12)
ax.set_title(
    f'Galaxy {PLOT_GAL_IDX}: x–y trajectory and search-sphere companions\n'
    r'$\bigstar$ = main galaxy  ●= companion  (colour = redshift, dashes = search radius)',
    fontsize=12,
)
ax.legend(fontsize=10, loc='upper left')
plt.tight_layout()
plt.show()

print(f"Tracked at {len(valid_si)} snapshots  |  "
      f"companions found: {len(comp_x)}  |  "
      f"unique companion snapshot count: {len(set(comp_si))}")

## 9 · Track diagnostics using `plot_main_galaxy_track` and `plot_neighborhood_track`

**Plot A** — main galaxy track only, as a connected line coloured by redshift, with periodic-boundary unwrapping.  
Same logic as `sfh_caesar.HDF5BuildHistory` but in position space. Use this to verify that the progenitor file is following the right object.

**Plot B** — same track plus every galaxy within 500 kpc/h (`radius_mpc=0.5`) at each snapshot.  
Companions recorded in `galaxy._progenitors` (from the search-radius scan in §3) are overplotted as distinct markers so you can check visual consistency: major-merger companions → red pentagons, minor → orange diamonds.

In [ ]:
from simbanator.visualization.plots import plot_main_galaxy_track

# Galaxy to inspect — default: the one with the most recorded companions
DIAG_GAL_IDX = max(galaxies, key=lambda k: len(galaxies[k].progenitors))
print(f'Inspecting galaxy index {DIAG_GAL_IDX} '
      f'({len(galaxies[DIAG_GAL_IDX].progenitors)} companion entries)')

fig, ax = plot_main_galaxy_track(
    galaxies[DIAG_GAL_IDX],
    caesar_paths,
    redshifts,
    box_size=BOX_SIZE,
    projection=('x', 'y'),
    title=f'Galaxy {DIAG_GAL_IDX} — main galaxy track (x-y)',
)
plt.tight_layout()
plt.show()

In [ ]:
from simbanator.visualization.plots import plot_neighborhood_track

fig, ax = plot_neighborhood_track(
    galaxies[DIAG_GAL_IDX],
    caesar_paths,
    redshifts,
    box_size=BOX_SIZE,
    radius_mpc=0.5,           # ≈ 500 kpc/h
    projection=('x', 'y'),
    mass_threshold=MASS_THRESHOLD,
    mass_threshold_maj=MASS_RATIO_MAJ,
    mass_threshold_min=MASS_RATIO_MIN,
    title=f'Galaxy {DIAG_GAL_IDX} — 500 kpc/h neighborhood + merger events',
)
plt.tight_layout()
plt.show()